# Tetris Hyperparameter Sensitivity Analysis

Standalone analysis for reviewer response. This notebook does not alter the main experiment flow.


In [ ]:
# Packages, seeds and paths
from edge_sim_py import *
import math
import os
import random
import msgpack
import pandas as pd
import json
import numpy as np
from numpy import random
from random import seed
import matplotlib.pyplot as plt
import seaborn as sns

abspath = os.path.abspath(os.path.join('..'))
algo_name = "tetris_weighted_sensitivity"

seed_list = [428956419, 1954324947, 1145661099, 1835732737, 794161987, 1329531353, 200496737, 633816299, 1410143363, 1282538739, 1029384756, 564738291, 918273645, 675849302, 123456789, 987654321, 246813579, 135792468, 1123581321, 314159265, 271828182, 161803398, 141421356, 173205080, 223606797, 707106781, 866025403, 577215664, 299792458, 602214076]
# Representative stress scenario. Expand this list if you want a broader but much slower sensitivity run.
scenario_list = ['2ec_high_off']

baseline_weights = {
    'alpha': 1.0,
    'beta': 1.0,
    'gamma': 1.0,
    'lambda_t': 1.0,
    'omega': 1.0,
    'rho': 1.0,
    'eta': 1.0,
}
weight_values = [0.25, 0.5, 1.0, 2.0, 4.0]


In [ ]:
# change the slas
def change_slas(path, mu, sigma):
    # Opening JSON file
    f = open(path)
    
    # returns JSON object as 
    # a dictionary
    data = json.load(f)
    
    # Iterating through the json
    # sla values from probabilistic distribution
    n = len(data['User'])
    sla_values = np.around(np.random.normal(mu, sigma, n), decimals=0)
    # list
    for i in range(len(data['User'])):
        print('>>> user', i, "<<<")
        print('before:', data['User'][i].get('attributes').get('delay_slas'))
        data['User'][i].get('attributes').get('delay_slas').update({str(i+1): sla_values[i]})
        print('after:', data['User'][i].get('attributes').get('delay_slas'))

    # Closing file
    f.close()

    # Serializing json
    json_object = json.dumps(data, indent=4)
    
    # Writing to sample.json
    with open(path, "w") as outfile:
        outfile.write(json_object)

# change the network delays
def change_network(path, link_delay, wireless_delay):
    # Opening JSON file
    f = open(path)
    
    # returns JSON object as 
    # a dictionary
    data = json.load(f)
    
    # Iterating through the json - networklink
    for i in range(len(data['NetworkLink'])):
        print('>>> network link', i, "<<<")
        print('before:', data['NetworkLink'][i].get('attributes').get('delay'))
        data['NetworkLink'][i].get('attributes').update({'delay': link_delay})
        print('after:', data['NetworkLink'][i].get('attributes').get('delay'))

    # Iterating through the json - wireless
    for i in range(len(data['BaseStation'])):
        print('>>> base station', i, "<<<")
        print('before:', data['BaseStation'][i].get('attributes').get('wireless_delay'))
        data['BaseStation'][i].get('attributes').update({'wireless_delay': wireless_delay})
        print('after:', data['BaseStation'][i].get('attributes').get('wireless_delay'))
    # Closing file
    f.close()

    # Serializing json
    json_object = json.dumps(data, indent=4)
    
    # Writing to sample.json
    with open(path, "w") as outfile:
        outfile.write(json_object)


In [ ]:
# Custom collect method to measure users
def user_custom_collect_method(self) -> dict: 
    # Python libraries
    import copy
    """Method that collects a set of metrics for the object.

    Returns:
        metrics (dict): Object metrics.
    """
    access_history = {}
    for app in self.applications:
        access_history[str(app.id)] = self.access_patterns[str(app.id)].history

    try: 
        delay_sla_deadlines = self.delay_slas.get(str(self.id)) - self.delays.get(str(self.id)) 
    except: 
         delay_sla_deadlines = None

    metrics = {
        "Instance ID": self.id,
        "Coordinates": self.coordinates,
        "Base Station": f"{self.base_station} ({self.base_station.coordinates})" if self.base_station else None,
        "Delays": self.delays.get(str(self.id)), #copy.deepcopy(self.delays),
        "Delay slas": self.delay_slas.get(str(self.id)),#copy.deepcopy(self.delay_slas),
        "Delay sla deadlines": delay_sla_deadlines,
        "Communication Paths": copy.deepcopy(self.communication_paths),
        "Making Requests": copy.deepcopy(self.making_requests),
        "Access History": copy.deepcopy(access_history)
    }
    return metrics

def server_custom_collect_method(self) -> dict:
        """Method that collects a set of metrics for the object.

        Returns:
            metrics (dict): Object metrics.
        """
        metrics = {
            "Instance ID": self.id,
            "Coordinates": self.coordinates,
            "Available": self.available,
            "CPU": self.cpu,
            "RAM": self.memory,
            "Disk": self.disk,
            "CPU Demand": self.cpu_demand,
            "RAM Demand": self.memory_demand,
            "Disk Demand": self.disk_demand,
            "Ongoing Migrations": self.ongoing_migrations,
            "Services": [service.id for service in self.services],
            "Registries": [registry.id for registry in self.container_registries],
            "Layers": [layer.instruction for layer in self.container_layers],
            "Images": [image.name for image in self.container_images],
            "Download Queue": [f.metadata["object"].instruction for f in self.download_queue],
            "Waiting Queue": [layer.instruction for layer in self.waiting_queue],
            "Max. Concurrent Layer Downloads": self.max_concurrent_layer_downloads,
            "Power Consumption": self.get_power_consumption(),
            "rfc": self.rfc,
            "rf_cpu": self.rf_cpu,
            "rf_memory": self.rf_memory,
        }
        return metrics


In [ ]:
## My min max scaler
def my_scaler(x_unscaled, x_min=None, x_max=None):
    if x_min == None:
        x_min=x_unscaled.min()
    if x_max == None:
        x_max=x_unscaled.max()
    x_scaled = (x_unscaled - x_min)/(x_max - x_min)
    return x_scaled


In [ ]:
# Weighted Tetris variant used only for sensitivity analysis

def find_shortest_path(origin_network_switch: object, target_network_switch: object) -> int:
    import networkx as nx
    topology = origin_network_switch.model.topology

    if not hasattr(topology, "delay_shortest_paths"):
        topology.delay_shortest_paths = {}

    key = (origin_network_switch, target_network_switch)
    if key in topology.delay_shortest_paths.keys():
        return topology.delay_shortest_paths[key]

    path = nx.shortest_path(G=topology, source=origin_network_switch, target=target_network_switch, weight="delay")
    topology.delay_shortest_paths[key] = path
    return path


def normalize_value(value, values):
    values = list(values)
    min_value = min(values)
    max_value = max(values)
    if min_value == max_value:
        return 0
    return (value - min_value) / (max_value - min_value)


def make_weighted_tetris_algorithm(weights):
    def weighted_tetris(parameters):
        print(f"[STEP {parameters['current_step']}] weights={weights}")

        for edge_server in EdgeServer.all():
            edge_server.rfc = 0
            edge_server.rf_cpu = 0
            edge_server.rf_memory = 0

        app_metadata = []
        for app in Application.all():
            user = app.users[0]
            delay_value = user.delays.get(str(app.id))
            if delay_value is None:
                delay_value = 0
            delay_sla = user.delay_slas[str(app.id)]
            deadline_margin = delay_sla - delay_value
            processing_proxy = sum(service.cpu_demand + service.memory_demand / 1024 for service in app.services)
            communication_proxy = delay_value
            phi = (
                weights['alpha'] * deadline_margin
                + weights['beta'] * processing_proxy
                + weights['gamma'] * communication_proxy
            )
            app_metadata.append({'object': app, 'phi': phi})

        app_metadata = sorted(app_metadata, key=lambda item: item['phi'])

        for app_item in app_metadata:
            app = app_item['object']
            user = app.users[0]
            user_switch = user.base_station.network_switch
            delay_sla = user.delay_slas[str(app.id)]

            for service in app.services:
                if service.being_provisioned:
                    continue

                service.drop = True
                candidates = []
                for edge_server in EdgeServer.all():
                    if not edge_server.available:
                        continue

                    topology = user_switch.model.topology
                    path = find_shortest_path(origin_network_switch=user_switch, target_network_switch=edge_server.network_switch)
                    path_delay = topology.calculate_path_delay(path=path)
                    has_capacity = edge_server.has_capacity_to_host(service=service)
                    projected_violation = max(0, path_delay - delay_sla)
                    drop_penalty = 0 if has_capacity else 1
                    projected_power = edge_server.get_power_consumption()

                    cpu_after = edge_server.cpu - edge_server.cpu_demand - service.cpu_demand
                    memory_after = edge_server.memory - edge_server.memory_demand - service.memory_demand
                    disk_after = edge_server.disk - edge_server.disk_demand - service.disk_demand
                    capacity_metric = ((max(cpu_after, 0) + 1) * (max(memory_after, 0) + 1) * (max(disk_after, 0) + 1)) ** (1 / 3)

                    objective = (
                        weights['lambda_t'] * projected_violation
                        + weights['omega'] * path_delay
                        + weights['rho'] * drop_penalty
                        + weights['eta'] * projected_power
                        + capacity_metric
                    )
                    candidates.append({
                        'object': edge_server,
                        'has_capacity': has_capacity,
                        'objective': objective,
                    })

                candidates = sorted(candidates, key=lambda item: (not item['has_capacity'], item['objective']))

                for candidate in candidates:
                    edge_server = candidate['object']
                    if candidate['has_capacity']:
                        service.drop = False
                        if service.server != edge_server:
                            service.provision(target_server=edge_server)
                        break

                    if ((edge_server.cpu - service.cpu_demand) >= 0) | ((edge_server.memory - service.memory_demand) >= 0):
                        edge_server.rfc = edge_server.rfc + 1
                        if ((edge_server.cpu - service.cpu_demand) >= 0):
                            edge_server.rf_cpu = edge_server.rf_cpu + (edge_server.cpu - service.cpu_demand)
                        if ((edge_server.memory - service.memory_demand) >= 0):
                            edge_server.rf_memory = edge_server.rf_memory + (edge_server.memory - service.memory_demand)

    return weighted_tetris


def stopping_criterion_services(model: object):
    provisioned_services = 0
    for service in Service.all():
        if service.server != None:
            provisioned_services += 1

    stop = False
    if (provisioned_services == Service.count()) | (model.schedule.steps == 60):
        stop = True
    return stop


def stopping_criterion_steps(model: object):
    n = 60
    return model.schedule.steps == n


In [ ]:
# Build one-at-a-time sensitivity configurations
weight_configs = []
for parameter in baseline_weights:
    for value in weight_values:
        weights = baseline_weights.copy()
        weights[parameter] = value
        config_id = f"{parameter}_{str(value).replace('.', 'p')}"
        weight_configs.append({'config_id': config_id, 'parameter': parameter, 'value': value, 'weights': weights})

pd.DataFrame([{k: v for k, v in item.items() if k != 'weights'} for item in weight_configs])


In [ ]:
# Run sensitivity simulations
all_server_logs = []
all_user_logs = []

for config in weight_configs:
    print(f"====== [CONFIG {config['config_id']}] ======")
    weighted_algorithm = make_weighted_tetris_algorithm(config['weights'])

    for seed_value in seed_list:
        print(f"====== [SEED {seed_value}] ======")
        seed(seed_value)
        np.random.seed(seed_value)

        for scenario in scenario_list:
            print(f">>> [scenario {scenario}] <<<")
            mu, sigma = (40, 2.5) if scenario == 'sample' else (15, 5)
            stopping_criterion = stopping_criterion_steps if scenario == 'sample' else stopping_criterion_services

            simulator = Simulator(
                tick_duration=1,
                tick_unit="seconds",
                stopping_criterion=stopping_criterion,
                resource_management_algorithm=weighted_algorithm,
                logs_directory=f"logs/algorithm={algo_name}/config={config['config_id']}/seed={seed_value}/scenario={scenario}",
            )

            path = f"{abspath}/datasets/dataset_{scenario}.json"
            change_slas(path, mu, sigma)
            simulator.initialize(input_file=f"{abspath}/datasets/dataset_{scenario}.json")
            User.collect = user_custom_collect_method
            EdgeServer.collect = server_custom_collect_method
            simulator.run_model()

            server_log = pd.DataFrame(simulator.agent_metrics["EdgeServer"])
            server_log['config_id'] = config['config_id']
            server_log['parameter'] = config['parameter']
            server_log['value'] = config['value']
            server_log['scenario'] = scenario
            server_log['seed'] = seed_value
            all_server_logs.append(server_log)

            user_log = pd.DataFrame(simulator.agent_metrics["User"])
            user_log['config_id'] = config['config_id']
            user_log['parameter'] = config['parameter']
            user_log['value'] = config['value']
            user_log['scenario'] = scenario
            user_log['seed'] = seed_value
            all_user_logs.append(user_log)

sensitivity_server = pd.concat(all_server_logs, ignore_index=True)
sensitivity_user = pd.concat(all_user_logs, ignore_index=True)
sensitivity_server.to_csv('results/tetris_hyper_sensitivity_server_raw.csv', index=False)
sensitivity_user.to_csv('results/tetris_hyper_sensitivity_user_raw.csv', index=False)


In [ ]:
# Summarize SLA violations and RFI
sensitivity_user['sla_violations'] = 0
sensitivity_user.loc[sensitivity_user['Delay sla deadlines'] < 0, 'sla_violations'] = 1
sensitivity_user['cumulative_sla_violations'] = sensitivity_user.groupby(['config_id', 'scenario', 'seed'])['sla_violations'].cumsum()

server = sensitivity_server.loc[(sensitivity_server['Available']) & (sensitivity_server['CPU Demand'] > 0)].reset_index(drop=True).copy()
server_edge = server.loc[server['Instance ID'] != 7].reset_index(drop=True).copy()
def scale_series(series):
    min_value = series.min()
    max_value = series.max()
    if min_value == max_value:
        return series * 0
    return (series - min_value) / (max_value - min_value)

server_edge['rf_cpu_norm'] = server_edge.groupby(['config_id', 'scenario'])['rf_cpu'].transform(scale_series)
server_edge['rf_memory_norm'] = server_edge.groupby(['config_id', 'scenario'])['rf_memory'].transform(scale_series)
server_edge['RFI'] = server_edge[['rf_cpu_norm', 'rf_memory_norm']].mean(axis=1)

sla_summary = sensitivity_user.groupby(['config_id', 'parameter', 'value', 'scenario', 'seed'])['cumulative_sla_violations'].max().reset_index()
rfi_summary = server_edge.groupby(['config_id', 'parameter', 'value', 'scenario', 'seed'])['RFI'].mean().reset_index()
summary = sla_summary.merge(rfi_summary, how='left', on=['config_id', 'parameter', 'value', 'scenario', 'seed'])

summary_stats = summary.groupby(['parameter', 'value', 'scenario']).agg(
    sla_mean=('cumulative_sla_violations', 'mean'),
    sla_std=('cumulative_sla_violations', 'std'),
    rfi_mean=('RFI', 'mean'),
    rfi_std=('RFI', 'std'),
).reset_index()

summary.to_csv('results/tetris_hyper_sensitivity_by_seed.csv', index=False)
summary_stats.to_csv('results/tetris_hyper_sensitivity_summary.csv', index=False)
summary_stats


In [ ]:
# Visualization: parameter impact on SLA violations and RFI
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.lineplot(data=summary_stats, x='value', y='sla_mean', hue='parameter', marker='o', ax=axes[0])
axes[0].set_xscale('log')
axes[0].set_title('SLA violations sensitivity')
axes[0].set_xlabel('Weight multiplier')
axes[0].set_ylabel('Mean SLA violations')

sns.lineplot(data=summary_stats, x='value', y='rfi_mean', hue='parameter', marker='o', ax=axes[1])
axes[1].set_xscale('log')
axes[1].set_title('RFI sensitivity')
axes[1].set_xlabel('Weight multiplier')
axes[1].set_ylabel('Mean RFI')

plt.tight_layout()
plt.savefig('results/tetris_hyper_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()
